## Installing Libraries

In [1]:
!pip install langgraph langchain langchain-openai langchain-community faiss-cpu pypdf langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: openai
    Found existing installation: openai 2.43.0
    Uninstalling openai-2.43.0:
      Successfully 

In [2]:
import os
from google.colab import userdata
OpenAI_API = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = OpenAI_API

## Import Libraries

In [3]:
from typing import TypedDict, Literal, List

from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import SentenceTransformersTokenTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document

from langgraph.graph import StateGraph, END

# Set the llm and embedding model
llm = ChatOpenAI(model="gpt-4o-mini")
embed_model = OpenAIEmbeddings(model="text-embedding-3-small")

/tmp/ipykernel_593/2182877414.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## Download and load Data

In [4]:
# Create directory to store pdfs in it
!mkdir data
%cd data

/content/data


In [5]:
!wget https://ocw.mit.edu/courses/1-264j-database-internet-and-systems-integration-technologies-fall-2013/d549b3ecf40310a93fec5da29a293fd5_MIT1_264JF13_lect_15.pdf
!wget https://ocw.mit.edu/courses/18-05-introduction-to-probability-and-statistics-spring-2022/mit18_05_s22_probability.pdf

--2026-07-16 06:01:45--  https://ocw.mit.edu/courses/1-264j-database-internet-and-systems-integration-technologies-fall-2013/d549b3ecf40310a93fec5da29a293fd5_MIT1_264JF13_lect_15.pdf
Resolving ocw.mit.edu (ocw.mit.edu)... 151.101.194.132, 151.101.66.132, 151.101.2.132, ...
Connecting to ocw.mit.edu (ocw.mit.edu)|151.101.194.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 160716 (157K) [application/pdf]
Saving to: ‘d549b3ecf40310a93fec5da29a293fd5_MIT1_264JF13_lect_15.pdf’

d549b3ecf40310a93fe 100%[===================>] 156.95K  --.-KB/s    in 0.02s   

2026-07-16 06:01:45 (6.59 MB/s) - ‘d549b3ecf40310a93fec5da29a293fd5_MIT1_264JF13_lect_15.pdf’ saved [160716/160716]

--2026-07-16 06:01:45--  https://ocw.mit.edu/courses/18-05-introduction-to-probability-and-statistics-spring-2022/mit18_05_s22_probability.pdf
Resolving ocw.mit.edu (ocw.mit.edu)... 151.101.194.132, 151.101.66.132, 151.101.2.132, ...
Connecting to ocw.mit.edu (ocw.mit.edu)|151.101.194.132|:443

In [6]:
# Generate two vector stores, one for statistics and another for SQL, and expose each as a retriever.

def load_sql():
  loader = PyPDFLoader("/content/data/d549b3ecf40310a93fec5da29a293fd5_MIT1_264JF13_lect_15.pdf")
  documents = loader.load()
  splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=100)
  nodes = splitter.split_documents(documents)
  vector_store = FAISS.from_documents(nodes, embed_model)
  return vector_store.as_retriever()

def load_stats():
  loader = PyPDFLoader("/content/data/mit18_05_s22_probability.pdf")
  documents = loader.load()
  splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=100)
  nodes = splitter.split_documents(documents)
  vector_store = FAISS.from_documents(nodes, embed_model)
  return vector_store.as_retriever()

In [7]:
# Build the two retrievers. Each one is described the same way the LlamaIndex
# QueryEngineTool descriptions were written, so the router LLM can decide
# which one to use for a given query.

sql_retriever = load_sql()
stats_retriever = load_stats()

TOOL_DESCRIPTIONS = {
    "sql": "Useful for retrieving specific context related to SQL from the SQL lecture.",
    "stats": "Useful for retrieving specific context related to probability from the stats handbook.",
}

## Agentic Rag with Router (LangGraph)

`RouteDecision` + a structured-output LLM call is the LangGraph equivalent of LlamaIndex's
`LLMSingleSelector`: the LLM is asked to pick exactly one choice from a list of choices,
except instead of an index it returns a Pydantic-validated label (`"sql"` or `"stats"`).

In [8]:
# Shared graph state that flows between every node

class RouterState(TypedDict):
    query: str
    route: str
    context: List[Document]
    response: str

In [9]:
# Structured output schema used to force the LLM to choose exactly one tool,
# mirroring LLMSingleSelector's single-choice selection behaviour.

class RouteDecision(BaseModel):
    tool: Literal["sql", "stats"] = Field(
        description="The single most relevant tool for answering the user's query."
    )

router_llm = llm.with_structured_output(RouteDecision)

def route_query(state: RouterState) -> RouterState:
    choices = "\n".join(f"- {name}: {desc}" for name, desc in TOOL_DESCRIPTIONS.items())
    prompt = (
        "You are a router that selects exactly one tool to answer the user's query.\n"
        f"Available tools:\n{choices}\n\n"
        f"User query: {state['query']}"
    )
    decision = router_llm.invoke(prompt)
    print(f"[router] selected tool: {decision.tool}")
    return {"route": decision.tool}

In [10]:
# Retrieval nodes - one per tool. Only the node matching the router's
# decision will run, via the conditional edge defined below.

def retrieve_sql(state: RouterState) -> RouterState:
    docs = sql_retriever.invoke(state["query"])
    return {"context": docs}

def retrieve_stats(state: RouterState) -> RouterState:
    docs = stats_retriever.invoke(state["query"])
    return {"context": docs}

def select_retriever(state: RouterState) -> str:
    # Maps the router's decision to the corresponding retrieval node name
    return "retrieve_sql" if state["route"] == "sql" else "retrieve_stats"

In [11]:
# Final synthesis node - generates the answer from the retrieved context,
# equivalent to what the underlying LlamaIndex query engine did internally.

def generate(state: RouterState) -> RouterState:
    context_text = "\n\n".join(doc.page_content for doc in state["context"])
    prompt = (
        "Answer the question using only the context below. "
        "If the answer isn't in the context, say you don't know.\n\n"
        f"Context:\n{context_text}\n\n"
        f"Question: {state['query']}"
    )
    answer = llm.invoke(prompt)
    return {"response": answer.content}

In [12]:
# Wire the nodes into a graph: route -> (retrieve_sql | retrieve_stats) -> generate -> END

graph = StateGraph(RouterState)

graph.add_node("route_query", route_query)
graph.add_node("retrieve_sql", retrieve_sql)
graph.add_node("retrieve_stats", retrieve_stats)
graph.add_node("generate", generate)

graph.set_entry_point("route_query")

graph.add_conditional_edges(
    "route_query",
    select_retriever,
    {"retrieve_sql": "retrieve_sql", "retrieve_stats": "retrieve_stats"},
)

graph.add_edge("retrieve_sql", "generate")
graph.add_edge("retrieve_stats", "generate")
graph.add_edge("generate", END)

query_engine = graph.compile()

In [13]:
response = query_engine.invoke({"query": "What are transactions?"})
print(response["response"])

[router] selected tool: sql
Transactions are groups of operations that must be treated as an atomic unit, meaning either all operations are executed, or all are rolled back in case of an error. They typically involve starting a transaction, executing multiple actions (such as inserting an OrderHeader and updating related OrderDetail rows), and then either committing the transaction if everything succeeds or rolling it back if any issues arise. Transactions ensure data integrity and consistency during operations in databases.


In [14]:
response = query_engine.invoke({"query": "Explain Probability vs. Statistics"})
print(response["response"])

[router] selected tool: stats
Probability and statistics are closely related but distinct concepts. Probability is logically self-contained and focuses on the rules and structures governing uncertain events, allowing for the assessment of likelihoods given a known random process. For example, if you know the probability of a fair coin landing heads is 0.5, you can calculate the probability of getting 60 or more heads in 100 tosses with a specific answer.

In contrast, statistics involves applying probability to draw conclusions from data, which can be more complex and subjective. For instance, if you toss an unknown coin 100 times and observe 60 heads, as a statistician, your task is to infer information about the coin's fairness based on this outcome, where different statisticians might arrive at different conclusions due to the uncertainty and interpretation of the data.

In summary, probability deals with the theoretical aspects of chance events, while statistics deals with empirica

To allow using multiple tools at once,
change `RouteDecision.tool` to `List[Literal["sql", "stats"]]`, fan out to both retrieval nodes with
`Send` (from `langgraph.constants`) instead of a single conditional edge, and concatenate the
retrieved context from both branches before the `generate` node runs.